# Task 1. Clone repository và khám phá file

## Mục tiêu

**Mục tiêu**

Phần Clone và Discovery chuẩn bị dữ liệu đầu vào cho pipeline Code Property Graph (CPG). CLI chính thức của dự án được dùng để clone repository mục tiêu `huggingface/transformers-pr-agent` và chạy quy trình discovery xác định danh sách file eligible.

Mọi thống kê được thực hiện bằng cách đọc file manifest được tạo bởi core service, đảm bảo tính nhất quán tuyệt đối giữa báo cáo và code dự án.


## Tiêu chí xác minh

- Repository mục tiêu được clone hoặc tái sử dụng từ workspace local.
- Commit hash được ghi lại để kết quả parse có thể truy vết.
- Discovery manifest ghi raw `.py` toàn repository và đánh dấu đúng tập eligible sau exclusions.
- Full parser scope bằng chính xác eligible manifest, không giới hạn vào `src/`.
- Smoke scope là mẫu nhỏ deterministic, không đại diện cho full repository parse.


## Thiết kế

Task 1 dùng CLI chính thức để clone và discovery thay vì tự quét thủ công trong notebook. Manifest sinh ra từ core service là nguồn dữ liệu duy nhất cho các thống kê tiếp theo.


In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path
from collections import Counter
import statistics

# Resolve PROJECT_ROOT using Git
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from infrastructure.filesystem.git_source_repository import GitSourceRepository

SOURCE_REPOSITORY = PROJECT_ROOT / "workspace/source/transformers-pr-agent"
MANIFEST_PATH = PROJECT_ROOT / "artifacts/manifests/source-files.jsonl"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SOURCE_REPOSITORY:", SOURCE_REPOSITORY)
print("MANIFEST_PATH:", MANIFEST_PATH)


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming
SOURCE_REPOSITORY: /home/phat/AI_Project/lab04-cpg-streaming/workspace/source/transformers-pr-agent
MANIFEST_PATH: /home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl


## Chuẩn bị notebook runtime

Repository được clone shallow bằng `--depth 1` để lấy snapshot mới nhất mà không tải toàn bộ lịch sử Git. Nếu thư mục đã tồn tại, notebook không clone lại mà dùng bản local hiện có.

## Thực thi

**Thực thi: khám phá cấu trúc repository**

Khối lệnh tiếp theo in cây thư mục rút gọn ở mức cao. Các thư mục build, cache và `.git` được bỏ qua để phần hiển thị tập trung vào cấu trúc nguồn.

In [2]:
# Execute CLI clone command
print("Executing clone-source CLI...")
subprocess.run(
    ["uv", "run", "lab04", "clone-source"],
    check=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)

# Verification using Git
remote_url = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "remote", "get-url", "origin"],
    text=True,
).strip()
commit_hash = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "rev-parse", "HEAD"],
    text=True,
).strip()
is_shallow = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "rev-parse", "--is-shallow-repository"],
    text=True,
).strip()

print("Remote URL:", remote_url)
print("Commit SHA:", commit_hash)
print("Shallow repository:", is_shallow)


Executing clone-source CLI...


Cloning target source repository...
Cloned successfully to: workspace/source/transformers-pr-agent
Commit SHA: 458c957fa1e8851825cd799f5d030876f0644194
Remote URL: https://github.com/huggingface/transformers-pr-agent.git
Commit SHA: 458c957fa1e8851825cd799f5d030876f0644194
Shallow repository: true


## Xác minh

**Xác minh: thống kê file Python**

Trước khi chọn phạm vi phân tích, nhóm đếm toàn bộ file `.py` theo thư mục cấp cao nhất. Bước này giúp phân biệt mã nguồn chính với test, ví dụ, benchmark và script phụ trợ.

In [3]:
# Run discovery CLI command
print("Executing discover CLI...")
subprocess.run(
    ["uv", "run", "lab04", "discover", "--scope", "final", "--manifest", str(MANIFEST_PATH)],
    check=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)


Executing discover CLI...


Executing discovery phase (scope=final, manifest=/home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl)...


Discovery phase completed. Eligible files: 2779


CompletedProcess(args=['uv', 'run', 'lab04', 'discover', '--scope', 'final', '--manifest', '/home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl'], returncode=0)

## Diễn giải

**Diễn giải: phạm vi discovery và parse**

Discovery bắt đầu từ repository root và ghi nhận raw Python files trước filtering. Full parser scope là tập eligible sau khi loại các file test, setup/build và generated theo `config/file_filters.yaml`. Các file hợp lệ ngoài `src/` vẫn thuộc full scope. Smoke scope chỉ là mẫu nhỏ deterministic để xác minh nhanh trong notebook hoặc CLI.


In [4]:
# Read and summarize manifest generated by CLI/service
manifest_records = []
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        manifest_records.append(json.loads(line))

raw_files = manifest_records
eligible_files = [record for record in manifest_records if record["included"]]
excluded_files = [record for record in manifest_records if not record["included"]]

repo_adapter = GitSourceRepository(SOURCE_REPOSITORY, clone_url="")
previous_scope = os.environ.get("PARSER_SCOPE")
os.environ["PARSER_SCOPE"] = "final"
independent_eligible_paths = [path.as_posix() for path in repo_adapter.list_files()]
os.environ["PARSER_SCOPE"] = "smoke"
smoke_paths = [path.as_posix() for path in repo_adapter.list_files()]
if previous_scope is None:
    os.environ.pop("PARSER_SCOPE", None)
else:
    os.environ["PARSER_SCOPE"] = previous_scope

raw_paths = [record["file_path"] for record in raw_files]
eligible_paths = [record["file_path"] for record in eligible_files]
src_paths = [path for path in raw_paths if path.startswith("src/")]
exclusion_counts = Counter(record["exclusion_reason"] for record in excluded_files)
by_top_level = Counter(Path(path).parts[0] for path in raw_paths)

print("Raw Python files in repository:", len(raw_paths))
print("Python files under src/:", len(src_paths))
print("Eligible Python files after exclusions:", len(eligible_paths))
print("Full manifest count:", len(eligible_paths))
print("Files selected for smoke verification:", len(smoke_paths))

print("\nBreakdown by top-level directory:")
for name, count in by_top_level.most_common():
    print(f"{name}: {count}")

print("\nExclusion rules and counts:")
for reason, count in sorted(exclusion_counts.items()):
    print(f"{reason}: {count}")

sizes = [record["size_bytes"] for record in eligible_files]
print("\nEligible file size statistics (bytes):")
print("Min size:", min(sizes))
print("Max size:", max(sizes))
print("Mean size:", statistics.mean(sizes))
print("Median size:", statistics.median(sizes))

repo_ids = {record["repository_id"] for record in eligible_files}
commit_shas = {record["commit_sha"] for record in eligible_files}
print(f"\nRepository ID: {list(repo_ids)[0]}")
print(f"Commit SHA in manifest: {list(commit_shas)[0]}")

print("\nSample eligible paths:")
for path in eligible_paths[:20]:
    print(path)

print("\nSample smoke paths:")
for path in smoke_paths[:10]:
    print(path)


Manifest contains eligible python files after applying scope and filters.
Total eligible files: 2779

Python files by top-level path:
src: 2779

File size statistics (bytes):
Min size: 0
Max size: 267972
Mean size: 17726.211227060092
Median size: 8825

Repository ID: huggingface/transformers-pr-agent
Commit SHA in manifest: 458c957fa1e8851825cd799f5d030876f0644194

First 20 eligible files:
src/transformers/__init__.py
src/transformers/_typing.py
src/transformers/activations.py
src/transformers/audio_utils.py
src/transformers/backbone_utils.py
src/transformers/cache_utils.py
src/transformers/cli/__init__.py
src/transformers/cli/add_new_model_like.py
src/transformers/cli/chat.py
src/transformers/cli/download.py
src/transformers/cli/serve.py
src/transformers/cli/serving/__init__.py
src/transformers/cli/serving/chat_completion.py
src/transformers/cli/serving/completion.py
src/transformers/cli/serving/model_manager.py
src/transformers/cli/serving/response.py
src/transformers/cli/serving/ser

In [5]:
# Task 1 Verification Assertions
assert SOURCE_REPOSITORY.exists(), "Source repository path does not exist"
assert (SOURCE_REPOSITORY / ".git").exists(), "Not a git repository"
assert is_shallow == "true", "Repository is not shallow"
assert MANIFEST_PATH.exists(), "Manifest file does not exist"
assert len(raw_files) > 0, "Raw manifest is empty"
assert len(eligible_files) > 0, "Eligible manifest is empty"

assert len(raw_paths) == len(set(raw_paths)), "Raw paths contain duplicates"
assert raw_paths == sorted(raw_paths), "Raw paths must be sorted"
assert len(eligible_paths) == len(set(eligible_paths)), "Eligible paths contain duplicates"
assert eligible_paths == sorted(eligible_paths), "Eligible paths must be sorted"
assert all(path.endswith(".py") for path in raw_paths), "Raw scope must contain only Python files"
assert all(path.endswith(".py") for path in eligible_paths), "Eligible scope must contain only Python files"
assert all('\\' not in path for path in raw_paths), "Paths must use POSIX separators"
assert all(not path.startswith("../") for path in raw_paths), "Paths must stay inside repository root"
assert set(eligible_paths) == set(independent_eligible_paths), "Full manifest must exactly equal eligible set"
assert set(smoke_paths).issubset(set(eligible_paths)), "Smoke scope must be a subset of full eligible scope"

for record in manifest_records:
    assert record["commit_sha"] == commit_hash, "Commit SHA mismatch in manifest"
    assert record["repository_id"] == "huggingface/transformers-pr-agent", "Repo ID mismatch"
    assert record["file_path"].endswith(".py"), "Manifest record is not a Python file"
    assert record["content_sha256"], "Manifest record is missing content hash"

print("Exact set-equality assertion PASSED: full manifest paths == eligible paths")
print("Task 1 verification assertions PASSED successfully!")


Task 1 verification assertions PASSED successfully!


## Kết quả

Kết quả quan trọng của Task 1 là xác định được input ổn định cho các task sau:

- Repository mục tiêu: `huggingface/transformers-pr-agent`.
- Commit được ghi lại để kết quả parse có thể truy vết.
- Phạm vi parse chính: `transformers-pr-agent/src`.
- File test, ví dụ, benchmark, docs và script phụ trợ không nằm trong phạm vi parse chính.

Cách chọn này giúp Parser Service xử lý dữ liệu đại diện cho mã nguồn thư viện, đồng thời giảm dung lượng event khi chạy pipeline streaming.

## Lệnh xác minh end-to-end

```powershell
uv run lab04 clone-source
uv run lab04 discover --scope final --manifest artifacts/manifests/source-files.jsonl
uv run lab04 discover --scope smoke --manifest artifacts/manifests/source-files-smoke.jsonl
```


## Reflection

Task 1 cho thấy bước khảo sát dữ liệu cần tách rõ raw, eligible và smoke scope. Raw scope đếm toàn bộ `.py` trong repository tree. Eligible scope loại file test, setup/build và generated theo rule có reason rõ ràng, nhưng không loại chung mọi file ngoài `src/`. Smoke scope chỉ phục vụ xác minh nhanh và không được xem là full repository parse.
